In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
# ----------------------------------------
# 1. Data Read
# ----------------------------------------
file_path = 'ChurnModeling.csv'  # Replace with the actual path to your CSV file
df = pd.read_csv(file_path)
print(df.head()) # Display the first few rows of the DataFrame

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [5]:
# ----------------------------------------
# 2. Feature/Target Split & Preprocessing
# ----------------------------------------
X = df.drop(['RowNumber','CustomerId','Surname','Exited'], axis=1)
y = df['Exited'].values

numeric_features     = ['CreditScore','Age','Tenure','Balance',
                        'NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary']
categorical_features = ['Geography','Gender']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(sparse_output=False, drop='first'), categorical_features)
])

X_proc = preprocessor.fit_transform(X)
X_proc

array([[-0.32622142,  0.29351742, -1.04175968, ...,  0.        ,
         0.        ,  0.        ],
       [-0.44003595,  0.19816383, -1.38753759, ...,  0.        ,
         1.        ,  0.        ],
       [-1.53679418,  0.29351742,  1.03290776, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.60498839, -0.27860412,  0.68712986, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.25683526,  0.29351742, -0.69598177, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.46377078, -1.04143285, -0.35020386, ...,  0.        ,
         0.        ,  0.        ]])

In [6]:
# ----------------------------------------
# 3. Train/Test Split & Tensor Conversion
# ----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


In [7]:
# ----------------------------------------
# 4. Dataset + DataLoader
# ----------------------------------------
class ChurnDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 2
train_loader = DataLoader(ChurnDataset(X_train, y_train),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(ChurnDataset(X_test, y_test),
                          batch_size=batch_size, shuffle=False)


In [21]:
# ----------------------------------------
# 5. Model Definition
# ----------------------------------------
class ChurnNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.model(x)


In [ ]:
# ----------------------------------------
# 5. Model Definition (with BatchNorm)
# ----------------------------------------
class ChurnNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.model(x)


In [22]:
# ----------------------------------------
# 6. Training & Evaluation Helpers
# ----------------------------------------
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        logits = model(Xb)
        loss   = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, X_eval, y_eval):
    model.eval()
    with torch.no_grad():
        logits = model(X_eval)
        preds  = torch.argmax(logits, dim=1)
    acc = accuracy_score(y_eval.numpy(), preds.numpy())
    report = classification_report(y_eval.numpy(), preds.numpy(),
                                   target_names=['Stayed','Exited'])
    return acc, report


In [23]:
# ----------------------------------------
# 7. Training Loop
# ----------------------------------------
def run_training(lr=0.01, epochs=50):
    input_dim = X_train.shape[1]
    model     = ChurnNet(input_dim)
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    
    for epoch in range(1, epochs+1):
        loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        if epoch % 10 == 0 or epoch==1:
            print(f"Epoch {epoch:2d}  Loss: {loss:.4f}")
    return model

# Train with default LR
print("Training with lr=0.01")
model = run_training(lr=0.01, epochs=50)


Training with lr=0.01
Epoch  1  Loss: 0.4338
Epoch 10  Loss: 0.3365
Epoch 20  Loss: 0.3308
Epoch 30  Loss: 0.3275
Epoch 40  Loss: 0.3242
Epoch 50  Loss: 0.3216


In [11]:
# ----------------------------------------
# 8. Evaluate on Test Set
# ----------------------------------------
acc, rpt = evaluate(model, X_test, y_test)
print(f"\nTest Accuracy: {acc:.4f}\n")
print(rpt)



Test Accuracy: 0.8680

              precision    recall  f1-score   support

      Stayed       0.88      0.97      0.92      1593
      Exited       0.81      0.46      0.58       407

    accuracy                           0.87      2000
   macro avg       0.84      0.72      0.75      2000
weighted avg       0.86      0.87      0.85      2000



In [17]:
# ----------------------------------------
# 10. Predict on a Single New Sample
# ----------------------------------------
sample = X_test[22].unsqueeze(0)
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(sample), dim=1).numpy().flatten()
    pred_class = int(probs.argmax())

print("\nSingle Sample Prediction:")
print(f"  Prob(stayed) = {probs[0]:.4f}")
print(f"  Prob(exited) = {probs[1]:.4f}")
print(f"  Predicted class: {'Exited' if pred_class==1 else 'Stayed'}")


Single Sample Prediction:
  Prob(stayed) = 0.9557
  Prob(exited) = 0.0443
  Predicted class: Stayed
